**Necessary imports**

In [1]:
import subprocess
import os
import pandas as pd

%load_ext autoreload
%autoreload 2

**Setting parameters**

In [2]:
simple_render_raw_data_dir = "/dlabdata1/tsoares/wikidumps/"
processed_data_dir = "/dlabdata1/boulenge/wikidumps_data/full_pages_experiment_all_langs"
processes = 48
max_samples_per_lang = 100
langs =  ["simple"] #sorted(set([filename.split("wiki")[0] for filename in os.listdir(simple_render_raw_data_dir) if "wiki" in filename]))
side_context_sizes = [5]  # Example context sizes
second_month = "20231001"
third_month = "20231101"

langs_to_remove = [] #["war", "ky", "nds-nl"]

for lang in langs_to_remove:
    langs.remove(lang)



**Defining data paths**

In [3]:
# Define directories
rev_files_dir = os.path.join(processed_data_dir, "revisions_files")
parsed_htmls_dir = os.path.join(processed_data_dir, "parsed_htmls")
contexts_splitted_data_dir = os.path.join(processed_data_dir, "contexts_splitted_data")
ready_to_score_data_dir = os.path.join(processed_data_dir, "ready_to_score_data")
scored_data_dir = os.path.join(processed_data_dir, "scored_data")
crawling_output_dir = os.path.join(processed_data_dir, "crawled_pages")

# Create necessary directories if they don't exist
os.makedirs(rev_files_dir, exist_ok=True)
os.makedirs(parsed_htmls_dir, exist_ok=True)
os.makedirs(contexts_splitted_data_dir, exist_ok=True)
os.makedirs(ready_to_score_data_dir, exist_ok=True)
os.makedirs(scored_data_dir, exist_ok=True)
os.makedirs(crawling_output_dir, exist_ok=True)

**Prepare files to be crawled and parsed**

In [4]:
for lang in langs:
    try:
        print(f"Processing language: {lang}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
            
        subprocess.run([
            "python", "generate_revisions_file.py",
            "--versions_file", f"{second_month_path}/eval/link_versions",
            "--html_dir", f"{second_month_path}/eval/pages",
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
            "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
            "--max_samples", str(max_samples_per_lang),
            "--save_path", save_path
        ])

        #revisions = pd.read_parquet(save_path)
        #display(revisions.head(5))
    except Exception as e:
        print(f"Error processing language {lang}: {e}")
        continue

Processing language: simple


Checking revisions: 100%|██████████| 1841/1841 [00:04<00:00, 392.07it/s]


**Crawling first versions only**

In [17]:
for lang in langs:
    print(f"Processing language: {lang}")
    subprocess.run([
        "python", "crawl.py",
        "--revisions_path", f"{rev_files_dir}/revisions_{lang}.parquet",
        "--output_dir", f"{crawling_output_dir}/{lang}",
        "--n_processes", str(processes),
        "--lang", lang,
        #"--crawl_second_versions", "False" (false by default)
    ])

Processing language: simple
Total number of pages to crawl: 100
Running crawling on 48 processes
Parallel multiprocess
Crawling finished, post processing...


Moving files: 100%|██████████| 100/100 [00:00<00:00, 154.35it/s]


**Parsing of the htmls**

In [6]:
for lang in langs:
    print(f"Processing language: {lang}")
    second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
    first_version_specific_html_dir = f"{crawling_output_dir}/{lang}"
    save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
    output_dir = f"{parsed_htmls_dir}/{lang}_v2"
    first_version_specific_output_dir = f"{parsed_htmls_dir}/{lang}_v1"
        
    subprocess.run([
        "python", "parsing.py",
        "--revisions_path", save_path,        
        "--html_dir", f"{second_month_path}/eval/pages",
        "--first_versions_specific_html_dir", first_version_specific_html_dir,
        "--output_dir", output_dir,
        "--first_versions_specific_output_dir", first_version_specific_output_dir,
        "--n_processes", str(processes)
    ])

    parsed_htmls_paths = os.listdir(first_version_specific_output_dir)
    sentences_path = list(filter(lambda x: x.endswith("sentences.jsonl"), parsed_htmls_paths))[0]
    links_path = list(filter(lambda x: x.endswith("links.jsonl"), parsed_htmls_paths))[0]
    sentences_df = pd.read_json(f"{first_version_specific_output_dir}/{sentences_path}")
    links_df = pd.read_json(f"{first_version_specific_output_dir}/{links_path}")

    display(sentences_df.head(5))
    display(links_df.head(5))   

Processing language: simple
First version html dir: /dlabdata1/boulenge/wikidumps_data/full_pages_experiment_all_langs/crawled_pages/simple
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1
Time taken for v1: 16.632322072982788
Writing sentences and links v2
Time taken for v2: 16.64988112449646


,md5,id,text,startIdx,endIdx,section,locList
0,587e3d3eb52163017e2ac83b2af970f1,0,Panvel also known as Gateway of Konkan is a ci...,40421,41055,Lead,"[{'start': 40421, 'end': 40427}, {'start': 404..."
1,a927954f8fe3f9f14b79beb2a3a3e6f9,1,"For the purposes of development and planning, ...",41056,41338,Lead,"[{'start': 41055, 'end': 41145}, {'start': 412..."
2,ab6927a2ac4d935afa081abdeb7ff149,2,It's worth noting that Panvel Municipal Corpor...,41339,41805,Lead,"[{'start': 41337, 'end': 41361}, {'start': 415..."
3,3d4267f15f6877a6b6dd24442a1a0616,3,Panvel is located about 40 kilometers east of ...,41875,42176,Location,"[{'start': 41874, 'end': 41875}, {'start': 418..."
4,982b1d089a20059861b54d638f3eb1f8,4,"The town is nestled amidst the Western Ghats, ...",42177,42315,Location,"[{'start': 42175, 'end': 42207}, {'start': 422..."


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/English_people,44388,English,44453,Lead,0
1,file:///wiki/Composer,44454,composer,44508,Lead,0
2,file:///wiki/Conductor_(music),44510,conductor,44603,Lead,0
3,file:///wiki/Music_producer,44608,producer,44694,Lead,0
4,file:///wiki/Movie_score,44725,music,44782,Lead,1


**Preprocessing**

In [7]:
for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
        output_dir = f"{contexts_splitted_data_dir}/{lang}/context_size_{side_context_size}"
        
        subprocess.run([
            "python", "modular_preprocessing.py",
            "--revisions_path", save_path,
            "--parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v2",
            "--first_versions_specific_parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v1",
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
            "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
            "--output_dir", output_dir,
            "--side_contexts_size", str(side_context_size),
            "--n_processes", str(processes)
        ])

        preprocessed_data_path = os.path.join(output_dir, f'test_data.parquet')

        preprocessed_data = pd.read_parquet(preprocessed_data_path)
        display(preprocessed_data.head(5))

Processing language: simple with side context size 5
Running on 48 processes
Parallel multiprocess


100%|██████████| 100/100 [00:04<00:00, 22.27it/s]


The length of the output is 372


100%|██████████| 372/372 [00:00<00:00, 16340.93it/s]


,source_title,source_ID,target_title,context,insert_indices,moved_insertion_marker,link_sentence,link_text,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,all_contexts
0,List_of_mayors_in_Norway_2023%E2%80%932027,1052251,M%C3%B8re_og_Romsdal,None,[26],None,"Hustadvika. KrF, and Centre Party; Nils Christ...",Møre og Romsdal,Lead,None,"In September and October 2023, every municipal...",9114883,9114892,None,missing_span,"[{'context': 'In September and October 2023, e..."
1,Patricia_Greene,818327,Birth_name,None,[6],None,Patricia Honor Richardson MBE (née Greene; bor...,née,Lead,None,Patricia Honor Greene MBE (born 1931) is an En...,9126470,9126510,None,missing_span,[{'context': 'Patricia Honor Greene MBE (born ...
2,Patricia_Greene,818327,Birth_name,None,None,None,Patricia Honor Greene was born in January 1931...,née,Early life,None,Patricia Honor Greene MBE (born 1931) is an En...,9126470,9126510,None,missing_section,[{'context': 'Patricia Honor Greene MBE (born ...
3,Patricia_Greene,818327,Birth_name,None,None,None,She is the long-standing voice of Jill Archer ...,née,Career,None,Patricia Honor Greene MBE (born 1931) is an En...,9126470,9126510,None,missing_section,[{'context': 'Patricia Honor Greene MBE (born ...
4,Patricia_Greene,818327,BBC_Radio_4,None,[6],None,Patricia Honor Richardson MBE (née Greene; bor...,BBC Radio 4,Lead,None,Patricia Honor Greene MBE (born 1931) is an En...,9126470,9126510,None,missing_span,[{'context': 'Patricia Honor Greene MBE (born ...


**Merging**

In [8]:
test_data_dir = contexts_splitted_data_dir
output_dir = ready_to_score_data_dir

subprocess.run([
    "python", "modular_merge_data.py",
    "--test_data_dir", test_data_dir,
    "--raw_data_dir", simple_render_raw_data_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--output_dir", output_dir,
    "--ignore_warnings"
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
scored_files_example = os.listdir(output_dir)
scored_files_example = list(filter(lambda x: (first_lang in x and first_context_size in x), scored_files_example))
final_input_data_path = os.path.join(output_dir, scored_files_example[0])

final_input_data_df = pd.read_parquet(final_input_data_path)
display(final_input_data_df.head(5))

Saving mention maps
Processing language simple (1/1)
Finished processing language simple
Processing language simple (1/1)
Loaded pages
Loaded redirects


100%|██████████| 238680/238680 [00:00<00:00, 862416.46it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 372 links for language simple
Removed 0 links because of empty contexts
Removed 39 links because target title is not in source titles
Removed 136 links because of missing section
After cleaning, there are 197 links for language simple
Finished processing language simple


,source_title,source_ID,target_title,context,insert_indices,moved_insertion_marker,link_sentence,link_text,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,all_contexts,target_lead
0,List_of_mayors_in_Norway_2023%E2%80%932027,1052251,M%C3%B8re_og_Romsdal,None,[26],None,"Hustadvika. KrF, and Centre Party; Nils Christ...",Møre og Romsdal,Lead,None,"In September and October 2023, every municipal...",9114883,9114892,None,missing_span,"[{'context': 'In September and October 2023, e...",Møre og Romsdal is a county in Norway. Norway ...
1,Patricia_Greene,818327,Birth_name,None,[6],None,Patricia Honor Richardson MBE (née Greene; bor...,née,Lead,None,Patricia Honor Greene MBE (born 1931) is an En...,9126470,9126510,None,missing_span,[{'context': 'Patricia Honor Greene MBE (born ...,A birth name is the family name someone has wh...
2,Patricia_Greene,818327,BBC_Radio_4,None,[6],None,Patricia Honor Richardson MBE (née Greene; bor...,BBC Radio 4,Lead,None,Patricia Honor Greene MBE (born 1931) is an En...,9126470,9126510,None,missing_span,[{'context': 'Patricia Honor Greene MBE (born ...,BBC Radio 4 is a domestic UK radio station tha...
3,Plow,38298,Tool,None,[2],None,A plow or plough is a machine used for several...,tool,Lead,None,Two animals pulling a plow with a farmer behin...,8833916,9150709,None,missing_mention_with_sentence_modifications,[{'context': 'Two animals pulling a plow with ...,\nTools are things used to make tasks easier. ...
4,Deaths_in_2023,986335,Burt_Young,None,[195],None,"Burt Young, 83, American actor (Rocky, Chinato...",Burt Young,October 2023,None,The following is a list of deaths that happene...,9138325,9138326,None,missing_sentence,[{'context': 'The following is a list of death...,Gerald Tommaso DeLouise better known by his st...


**Scoring the contexts**

In [18]:
data_dir = ready_to_score_data_dir
output_dir = scored_data_dir

subprocess.run([
    "python", "get_scores.py",
    "--data_dir", data_dir,
    "--output_dir", output_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--models_dir", "/dlabdata1/tsoares/models",
    "--models_prefix", "roberta_full",
    "--use_section_title",
    "--use_mentions",
    "--use_cuda",
    "--only_multilingual",
    "--multilingual_name", "multilingual-all"
])

scored_files_example = os.listdir(output_dir)[0]
scored_data = pd.read_parquet(f"{output_dir}/{scored_files_example}")

exemple = scored_data['all_contexts'][0][0]

for key in exemple.keys():
    print(f"{key}:\n{exemple[key]}\n")

KeyboardInterrupt: 

**Colouring the pages**

In [20]:
output_dir = os.path.join(processed_data_dir, "colored_pages")

# Run the coloring script
subprocess.run([
    "python", "color_pages.py",
    "--scored_data_dir", scored_data_dir,
    "--html_dir", crawling_output_dir,
    "--files_link_output_dir", output_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--extra_crawled_files_dir", crawling_output_dir,
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
colored_files_paths = os.path.join(output_dir, first_lang, f"first_context_size.json")

print(f"Colored files path: {colored_files_paths}")



**Locally download the colored pages**

In [ ]:
user_cluster  = "boulenge@iccluster039"
local_path = "Desktop/htmls"

scp_command = f"scp -qr {user_cluster}:{output_dir} {local_path} > /dev/null"

#subprocess.run(scp_command, shell=True)

print(f"Run the following in the terminal to dowload the colored pages:\n{scp_command}")

Run the following in the terminal to dowload the colored pages:
scp -qr boulenge@iccluster039:/dlabdata1/boulenge/wikidumps_data/full_pages_experiement2/colored_pages Desktop/htmls > /dev/null
